In [7]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


# 1. Load your actual dataset
# Since the notebook and the CSV are in the same folder, we can read it directly
df = pd.read_csv('IMDB Dataset.csv')

print("--- Dataset Shape ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
print("\n--- First 2 Rows of Raw Data ---")
print(df.head(2))

# 2. Define the text cleaning function
def clean_text(text):
    # Ensure the input is treated as a string
    text = str(text)
    
    # Remove HTML tags like <br />, <br>
    text = re.sub(r'<br\s*/?>', ' ', text)
    
    # Keep only letters and spaces (removes numbers, punctuation, special chars)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Tokenization: break sentences into individual words
    words = word_tokenize(text)
    
    # Stopwords removal (remove high frequency words like 'the', 'is', 'in')
    stop_words = set(stopwords.words('english'))
    cleaned_words = [word for word in words if word not in stop_words]
    
    # Glue the words back into a clean string sentence
    return " ".join(cleaned_words)

# 3. Clean the dataset (this might take a few seconds depending on dataset size)
print("\nCleaning reviews... Please wait...")
df['cleaned_review'] = df['review'].apply(clean_text)
print("Cleaning complete!")

print("\n--- Sample of Cleaned Text ---")
print(df[['review', 'cleaned_review', 'sentiment']].head(2))

--- Dataset Shape ---
Total Rows: 50000, Total Columns: 2

--- First 2 Rows of Raw Data ---
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive

Cleaning reviews... Please wait...
Cleaning complete!

--- Sample of Cleaned Text ---
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   

                                      cleaned_review sentiment  
0  one reviewers mentioned watching oz episode yo...  positive  
1  wonderful little production filming technique ...  positive  


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Split the data into Features (X) and Labels (y)
X = df['cleaned_review']
y = df['sentiment']

# 2. Split into Training set (80%) and Testing set (20%)
# This ensures we train our model on one group, and test it on a group it has never seen before.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

# 3. Initialize the TF-IDF Vectorizer
# max_features=2000 means we only keep the top 2000 most important words in the vocabulary
tfidf = TfidfVectorizer(max_features=2000)

# 4. Transform the text data into numbers
# Fit_transform calculates the vocabulary and converts X_train to numbers
X_train_tfidf = tfidf.fit_transform(X_train)
# Transform only converts X_test using the vocabulary learned from X_train
X_test_tfidf = tfidf.transform(X_test)

print("\n--- Feature Matrix Shape ---")
print(f"X_train_tfidf shape: {X_train_tfidf.shape}") 
# (400, 2000) means 400 reviews, each represented by a vector of 2000 numerical word-scores

Training samples: 40000
Testing samples: 10000

--- Feature Matrix Shape ---
X_train_tfidf shape: (40000, 2000)


In [3]:
from sklearn.linear_model import LogisticRegression

# 1. Initialize the Base Model
# Logistic Regression is fast, robust, and perfect for baseline text classification
model = LogisticRegression(random_state=42)

# 2. Train (Fit) the model on our training data
print("Training the base model classifier...")
model.fit(X_train_tfidf, y_train)
print("Model training complete!")

# 3. Quick sanity check: Check training score vs testing score
train_accuracy = model.score(X_train_tfidf, y_train)
print(f"\nAccuracy on Training Data: {train_accuracy * 100:.2f}%")

Training the base model classifier...
Model training complete!

Accuracy on Training Data: 89.47%


In [4]:
import joblib

joblib.dump(model, "sentiment_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [5]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Make predictions on the hidden test data
y_pred = model.predict(X_test_tfidf)

# 2. Calculate general accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print("==================================================")
print(f"🎯 BASE MODEL TEST ACCURACY: {test_accuracy * 100:.2f}%")
print("==================================================\n")

# 3. Print the detailed Classification Report
print("📋 Detailed NLP Classification Report:")
print(classification_report(y_test, y_pred))

# 4. Print the Confusion Matrix
print("🧩 Confusion Matrix Matrix (Raw Counts):")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Quick look at the alignment:
print(f"\nTrue Negative (Correctly predicted Negative): {cm[0][0]}")
print(f"False Positive (Negative marked as Positive): {cm[0][1]}")
print(f"False Negative (Positive marked as Negative): {cm[1][0]}")
print(f"True Positive (Correctly predicted Positive): {cm[1][1]}")

🎯 BASE MODEL TEST ACCURACY: 88.06%

📋 Detailed NLP Classification Report:
              precision    recall  f1-score   support

    negative       0.88      0.88      0.88      5000
    positive       0.88      0.88      0.88      5000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000

🧩 Confusion Matrix Matrix (Raw Counts):
[[4384  616]
 [ 578 4422]]

True Negative (Correctly predicted Negative): 4384
False Positive (Negative marked as Positive): 616
False Negative (Positive marked as Negative): 578
True Positive (Correctly predicted Positive): 4422


In [6]:
def predict_my_review(custom_review):
    # 1. Clean the incoming text using the function from Phase 1
    cleaned = clean_text(custom_review)
    
    # 2. Transform the text using the ALREADY FITTED tfidf vectorizer
    # Crucial: Use .transform(), NOT .fit_transform() here!
    numerical_features = tfidf.transform([cleaned])
    
    # 3. Predict using our trained model
    prediction = model.predict(numerical_features)[0]
    
    # 4. Get probability scores to see how confident the model is
    probabilities = model.predict_proba(numerical_features)[0]
    classes = model.classes_
    
    print(f'💬 Review: "{custom_review}"')
    print(f'🤖 Predicted Sentiment: {prediction.upper()}')
    print(f'📊 Confidence: {classes[0]} = {probabilities[0]*100:.1f}% | {classes[1]} = {probabilities[1]*100:.1f}%\n')

# --- TEST IT OUT ---
predict_my_review("This movie was an absolute masterpiece! I sat on the edge of my seat the whole time.")
predict_my_review("Total garbage. The script was terrible and the actors looked bored.")
predict_my_review("It had some good visuals, but the plot fell completely flat.")

💬 Review: "This movie was an absolute masterpiece! I sat on the edge of my seat the whole time."
🤖 Predicted Sentiment: POSITIVE
📊 Confidence: negative = 12.4% | positive = 87.6%

💬 Review: "Total garbage. The script was terrible and the actors looked bored."
🤖 Predicted Sentiment: NEGATIVE
📊 Confidence: negative = 99.9% | positive = 0.1%

💬 Review: "It had some good visuals, but the plot fell completely flat."
🤖 Predicted Sentiment: NEGATIVE
📊 Confidence: negative = 87.8% | positive = 12.2%

